<a href="https://colab.research.google.com/github/codylewis1286-spec/Mouse-MRI/blob/main/ANTs_works95__Human_Skull_Stripper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Human Skull Stripper — IXI Dataset
**Pipeline:** N4 Bias Correction → T2→T1 Rigid Registration → Deep Learning Brain Extraction (ANTsPyNet) → Morphological Cleanup

**Runtime requirement:** GPU (T4 or better). Set via `Runtime → Change runtime type → T4 GPU`

---
## Block 1: Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Local scratch directories for intermediate files
os.makedirs('/content/data/ixi_raw',       exist_ok=True)
os.makedirs('/content/data/ixi_processed', exist_ok=True)

print('✓ Drive mounted.')
print('✓ Scratch directories ready.')

Mounted at /content/drive
✓ Drive mounted.
✓ Scratch directories ready.


---
## Block 2: Install Dependencies
`antspyx` — image I/O, N4 correction, registration  
`antspynet` — deep learning brain extraction (the correct tool for this job)

In [ ]:
# These are the correct, installable packages that actually work
!pip install -q antspyx antspynet

import ants
import antspynet
import torch
import nibabel as nib
import numpy as np
import importlib.metadata

print(f'✓ ANTsPy     : {ants.__version__}')
print(f'✓ ANTsPyNet  : {importlib.metadata.version("antspynet")}')
print(f'✓ PyTorch    : {torch.__version__}')
print(f'✓ CUDA ready : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✓ GPU        : {torch.cuda.get_device_name(0)}')

✓ ANTsPy     : 0.6.3
✓ ANTsPyNet  : 0.3.2
✓ PyTorch    : 2.11.0+cu128
✓ CUDA ready : True
✓ GPU        : Tesla T4


---
## Block 3: Import All Libraries

In [ ]:
import os
import glob
import random

import ants
import antspynet
import torch
import nibabel as nib
import numpy as np
from scipy.ndimage import label, binary_fill_holes

print('✓ All libraries loaded.')

✓ All libraries loaded.


---
## Block 4: Skull Stripping Pipeline Function

**Steps per subject:**
1. Load T1 + T2
2. N4 bias field correction on both
3. Rigid registration: T2 → T1 space
4. Deep learning brain extraction on T1 via `antspynet.brain_extraction()`
5. Morphological cleanup: hole filling + largest connected component
6. Apply mask to both T1 and T2, save outputs

In [ ]:
def process_single_subject(t1_raw_path, t2_raw_path, output_dir, subject_id):
    """
    Full skull stripping pipeline for a paired T1/T2 subject.

    Args:
        t1_raw_path : str  - Path to raw T1 NIfTI file
        t2_raw_path : str  - Path to raw T2 NIfTI file
        output_dir  : str  - Directory to write processed outputs
        subject_id  : str  - Subject identifier prefix (e.g. 'IXI013')
    """
    os.makedirs(output_dir, exist_ok=True)

    out_mask    = os.path.join(output_dir, f'{subject_id}_brain_mask.nii.gz')
    out_t1      = os.path.join(output_dir, f'{subject_id}_t1_stripped.nii.gz')
    out_t2      = os.path.join(output_dir, f'{subject_id}_t2_stripped.nii.gz')

    print(f'\n{"="*60}')
    print(f'  SUBJECT: {subject_id}')
    print(f'{"="*60}')

    # ------------------------------------------------------------------
    # Step 1: Load raw volumes
    # ------------------------------------------------------------------
    print('  [1/5] Loading volumes...')
    t1_img = ants.image_read(t1_raw_path)
    t2_img = ants.image_read(t2_raw_path)

    # ------------------------------------------------------------------
    # Step 2: N4 Bias Field Correction
    # Removes low-frequency intensity inhomogeneity from scanner field
    # ------------------------------------------------------------------
    print('  [2/5] N4 Bias Field Correction...')
    t1_n4 = ants.n4_bias_field_correction(t1_img)
    t2_n4 = ants.n4_bias_field_correction(t2_img)

    # ------------------------------------------------------------------
    # Step 3: Rigid Registration — T2 into T1 space
    # 6-DOF mutual information registration
    # ------------------------------------------------------------------
    print('  [3/5] Rigid co-registration T2 → T1...')
    reg        = ants.registration(
                     fixed=t1_n4,
                     moving=t2_n4,
                     type_of_transform='Rigid'
                 )
    t2_aligned = reg['warpedmovout']

    # ------------------------------------------------------------------
    # Step 4: Deep Learning Brain Extraction
    # antspynet.brain_extraction() — contrast-agnostic, GPU-accelerated
    # Returns a probability map (0.0–1.0); threshold at 0.5 for binary mask
    # ------------------------------------------------------------------
    print('  [4/5] Deep learning brain extraction (ANTsPyNet)...')
    prob_mask  = antspynet.brain_extraction(t1_n4, modality='t1', verbose=False)
    mask_array = (prob_mask.numpy() > 0.5).astype(np.uint8)

    # ------------------------------------------------------------------
    # Step 5: Morphological Cleanup
    # Fill internal voids (ventricles showing as holes), then keep only
    # the largest connected 3D component (drops stray nasal/eye clusters)
    # ------------------------------------------------------------------
    print('  [5/5] Morphological cleanup + saving outputs...')

    # Fill holes
    mask_array = binary_fill_holes(mask_array).astype(np.uint8)

    # Largest connected component only
    labeled, n_components = label(mask_array)
    if n_components > 1:
        sizes          = np.bincount(labeled.ravel())
        sizes[0]       = 0                                   # ignore background
        mask_array     = (labeled == sizes.argmax()).astype(np.uint8)

    # Rebuild ANTs image preserving T1 geometry
    final_mask = ants.from_numpy(
        mask_array.astype(np.float32),
        origin    = t1_n4.origin,
        spacing   = t1_n4.spacing,
        direction = t1_n4.direction
    )

    # Apply mask and write outputs
    ants.image_write(final_mask,                               out_mask)
    ants.image_write(ants.mask_image(t1_n4,      final_mask), out_t1)
    ants.image_write(ants.mask_image(t2_aligned, final_mask), out_t2)

    print(f'  ✓ Done → {output_dir}')
    return out_mask, out_t1, out_t2


print('✓ Pipeline function defined.')

✓ Pipeline function defined.


---
## Block 5: Batch Processing Loop

**Expected Drive structure:**
```
MyDrive/
  Latent Diffusion Model/
    IXI Brains/
      <site>/
        NIfTI/
          IXI013-HH-1212-T1.nii.gz
          IXI013-HH-1212-T2.nii.gz
```

Update `RAW_DATA_DIR` and `PROCESSED_DATA_DIR` if your paths differ.

In [ ]:
# ── Path Configuration ──────────────────────────────────────────────────
RAW_DATA_DIR       = '/content/drive/MyDrive/Latent Diffusion Model/IXI Brains/'
PROCESSED_DATA_DIR = '/content/drive/MyDrive/Latent Diffusion Model/IXI Brains/ixi_processed_outputs/'
MAX_SUBJECTS       = 10   # Set to None to process everything
RANDOM_SAMPLE      = False # True = random subset; False = first N in sorted order
RANDOM_SEED        = 42
# ────────────────────────────────────────────────────────────────────────

t1_files = sorted(glob.glob(os.path.join(RAW_DATA_DIR, '**', 'NIfTI', '*-T1.nii.gz'), recursive=True))
t2_files = sorted(glob.glob(os.path.join(RAW_DATA_DIR, '**', 'NIfTI', '*-T2.nii.gz'), recursive=True))

print(f'Found {len(t1_files)} T1 files and {len(t2_files)} T2 files.')

# Pair by subject ID (strip modality suffix, match on base prefix)
def subject_id_from_path(p):
    return os.path.basename(p).replace('-T1.nii.gz', '').replace('-T2.nii.gz', '')

t1_map = {subject_id_from_path(p): p for p in t1_files}
t2_map = {subject_id_from_path(p): p for p in t2_files}

# Only keep subjects that have BOTH T1 and T2
common_ids = sorted(set(t1_map.keys()) & set(t2_map.keys()))
print(f'Complete T1+T2 pairs: {len(common_ids)}')

if len(common_ids) == 0:
    raise RuntimeError('No matching T1/T2 pairs found. Check RAW_DATA_DIR and your Drive structure.')

# Select batch
if RANDOM_SAMPLE:
    random.seed(RANDOM_SEED)
    selected_ids = random.sample(common_ids, min(MAX_SUBJECTS or len(common_ids), len(common_ids)))
else:
    selected_ids = common_ids[:MAX_SUBJECTS] if MAX_SUBJECTS else common_ids

print(f'Processing {len(selected_ids)} subjects: {selected_ids[:5]}{"..." if len(selected_ids) > 5 else ""}')

Found 5 T1 files and 5 T2 files.
Complete T1+T2 pairs: 5
Processing 5 subjects: ['IXI013-HH-1212', 'IXI015-HH-1258', 'IXI016-Guys-0697', 'IXI017-Guys-0698', 'IXI019-Guys-0702']


In [ ]:
# ── Run the batch ────────────────────────────────────────────────────────
results  = []
failures = []

for i, subject_id in enumerate(selected_ids):
    print(f'\n[{i+1}/{len(selected_ids)}]')
    try:
        out = process_single_subject(
            t1_raw_path = t1_map[subject_id],
            t2_raw_path = t2_map[subject_id],
            output_dir  = PROCESSED_DATA_DIR,
            subject_id  = subject_id
        )
        results.append({'id': subject_id, 'status': 'OK', 'outputs': out})
    except Exception as e:
        print(f'  ❌ FAILED: {e}')
        failures.append({'id': subject_id, 'error': str(e)})

# ── Summary ──────────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print(f'  BATCH COMPLETE')
print(f'  Succeeded : {len(results)}')
print(f'  Failed    : {len(failures)}')
if failures:
    print('  Failed subjects:')
    for f in failures:
        print(f'    - {f["id"]}: {f["error"]}')
print(f'{"="*60}')


[1/5]

  SUBJECT: IXI013-HH-1212
  [1/5] Loading volumes...
  [2/5] N4 Bias Field Correction...
  [3/5] Rigid co-registration T2 → T1...
  [4/5] Deep learning brain extraction (ANTsPyNet)...
5683832/5683832 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
14969865/14969865 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
  [5/5] Morphological cleanup + saving outputs...
  ✓ Done → /content/drive/MyDrive/Latent Diffusion Model/IXI Brains/ixi_processed_outputs/

[2/5]

  SUBJECT: IXI015-HH-1258
  [1/5] Loading volumes...
  [2/5] N4 Bias Field Correction...
  [3/5] Rigid co-registration T2 → T1...
  [4/5] Deep learning brain extraction (ANTsPyNet)...
  [5/5] Morphological cleanup + saving outputs...
  ✓ Done → /content/drive/MyDrive/Latent Diffusion Model/IXI Brains/ixi_processed_outputs/

[3/5]

  SUBJECT: IXI016-Guys-0697
  [1/5] Loading volumes...
  [2/5] N4 Bias Field Correction...
  [3/5] Rigid co-registration T2 → T1...
  [4/5] Deep learning brain extraction (ANTsPyNet)...
  [5/5] Morphological cleanup + saving o

  [5/5] Morphological cleanup + saving outputs...
  ✓ Done → /content/drive/MyDrive/Latent Diffusion Model/IXI Brains/ixi_processed_outputs/

  BATCH COMPLETE
  Succeeded : 5
  Failed    : 0


---
## Block 6: Quick Visual QC
Spot-check the first processed subject — displays axial/coronal/sagittal slices of stripped T1.

In [ ]:
import matplotlib.pyplot as plt

if results:
    _, t1_stripped_path, _ = results[0]['outputs']
    subject_id             = results[0]['id']

    vol  = ants.image_read(t1_stripped_path).numpy()
    cx, cy, cz = [s // 2 for s in vol.shape]  # centre slices

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f'QC — Skull-stripped T1: {subject_id}', fontsize=14)

    axes[0].imshow(vol[cx, :, :].T, cmap='gray', origin='lower'); axes[0].set_title('Sagittal')
    axes[1].imshow(vol[:, cy, :].T, cmap='gray', origin='lower'); axes[1].set_title('Coronal')
    axes[2].imshow(vol[:, :, cz].T, cmap='gray', origin='lower'); axes[2].set_title('Axial')

    for ax in axes:
        ax.axis('off')

    plt.tight_layout()
    plt.show()
else:
    print('No successful results to visualise.')